# 第 1 周第 1 天 —— 用 GPT 解读葡萄牙联赛积分数据

## 练习目标（理念）

准备一份 **Primeira Liga** 球队积分榜的「假数据 / 测试数据」，塞进 `user` 消息，让模型扮演体育分析师，用自然语言总结前几名球队的统计。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `.env` + API Key | `load_dotenv`、`OPENAI_API_KEY` 格式检查 |
| `system` / `user` | 体育分析师角色 + 问题 + `stats` 数据 |
| Chat Completions | `openai.chat.completions.create(...)` |
| 笔记本展示 | `display(Markdown(...))` |

## 怎么跑

1. 配置好 `.env` 中的 `OPENAI_API_KEY`，从上到下运行
2. `stats` 单元格是本地构造的字典列表，不依赖实时爬虫
3. 最后一格会把 `stats` 拼进 prompt 并调用 `gpt-5-nano`；可改 `user_prompt` 问别的问题（勿改 system/user 英文原意若要保持对比一致）


In [ ]:
# ========== 导入：环境变量、笔记本展示、OpenAI 客户端 ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里渲染模型回复
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 如果运行此单元时出现错误，请转到故障排除笔记本！


In [ ]:
# ========== 加载 .env 并检查 OPENAI_API_KEY ==========

# 加载环境变量：override=True 表示用 .env 覆盖已有同名变量
load_dotenv(override=True)
# 读取 API Key（字符串名必须与环境变量一致）
api_key = os.getenv('OPENAI_API_KEY')

# ========== 检查钥匙：缺 Key / 前缀不对 / 首尾空白 ==========

# 没读到 Key
if not api_key:
    # 排错文案保持英文原样
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# Key 不像常见的 sk-proj- 项目密钥
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 首尾有空格/Tab，容易导致请求失败
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 格式看起来正常
    print("API key found and looks good so far!")


In [ ]:
# ========== 测试用积分榜数据：本地伪造的球队统计（不依赖实时 API）==========
# 每条字典字段含义（Portuguese league 常见记法）：
# position=名次, team=队名, games_played=场次, wins/draws/losses=胜/平/负
# goals_for / goals_against / goal_difference=进球/失球/净胜球, points=积分
# recent_results：近几场结果字母（如 V/E/D），原数据保持不动，只作上下文喂给模型

stats = [{
    "position": 2,
    "team": "Sporting CP",
    "games_played": 13,
    "wins": 10,
    "draws": 2,
    "losses": 1,
    "goals_for": 32,
    "goals_against": 7,
    "goal_difference": 25,
    "points": 32,
    "recent_results": ["E", "V", "V", "V", "V"],
},
{
    "position": 3,
    "team": "Benfica",
    "games_played": 13,
    "wins": 8,
    "draws": 5,
    "losses": 0,
    "goals_for": 26,
    "goals_against": 8,
    "goal_difference": 18,
    "points": 29,
    "recent_results": ["E", "V", "E", "V", "V"],
},
{
    "position": 4,
    "team": "Gil Vicente",
    "games_played": 13,
    "wins": 7,
    "draws": 3,
    "losses": 3,
    "goals_for": 16,
    "goals_against": 6,
    "goal_difference": 10,
    "points": 24,
    "recent_results": ["D", "E", "V", "V", "V"],
},
{
    "position": 5,
    "team": "Braga",
    "games_played": 13,
    "wins": 6,
    "draws": 4,
    "losses": 3,
    "goals_for": 25,
    "goals_against": 12,
    "goal_difference": 13,
    "points": 22,
    "recent_results": ["V", "V", "V", "D", "V"],
},
{
    "position": 6,
    "team": "Famalicão",
    "games_played": 13,
    "wins": 5,
    "draws": 5,
    "losses": 3,
    "goals_for": 14,
    "goals_against": 9,
    "goal_difference": 5,
    "points": 20,
    "recent_results": ["D", "E", "D", "V", "V"],
},
{
    "position": 7,
    "team": "Moreirense",
    "games_played": 13,
    "wins": 6,
    "draws": 2,
    "losses": 5,
    "goals_for": 21,
    "goals_against": 20,
    "goal_difference": 1,
    "points": 20,
    "recent_results": ["E", "E", "D", "V", "D"],
},
{
    "position": 8,
    "team": "Vitória SC",
    "games_played": 13,
    "wins": 5,
    "draws": 3,
    "losses": 5,
    "goals_for": 14,
    "goals_against": 17,
    "goal_difference": -3,
    "points": 18,
    "recent_results": ["V", "V", "D", "D", "V"],
},
{
    "position": 9,
    "team": "Alverca",
    "games_played": 13,
    "wins": 5,
    "draws": 2,
    "losses": 6,
    "goals_for": 15,
    "goals_against": 19,
    "goal_difference": -4,
    "points": 17,
    "recent_results": ["V", "V", "E", "D", "D"],
},
{
    "position": 10,
    "team": "Rio Ave",
    "games_played": 13,
    "wins": 3,
    "draws": 7,
    "losses": 3,
    "goals_for": 17,
    "goals_against": 21,
    "goal_difference": -4,
    "points": 16,
    "recent_results": ["V", "E", "E", "D", "V"],
},
{
    "position": 11,
    "team": "Santa Clara",
    "games_played": 13,
    "wins": 4,
    "draws": 3,
    "losses": 6,
    "goals_for": 11,
    "goals_against": 14,
    "goal_difference": -3,
    "points": 15,
    "recent_results": ["V", "E", "D", "D", "V"],
},
{
    "position": 12,
    "team": "Estoril",
    "games_played": 13,
    "wins": 3,
    "draws": 5,
    "losses": 5,
    "goals_for": 22,
    "goals_against": 21,
    "goal_difference": 1,
    "points": 14,
    "recent_results": ["E", "D", "V", "V", "E"],
},
{
    "position": 13,
    "team": "Estrela da Amadora",
    "games_played": 13,
    "wins": 3,
    "draws": 5,
    "losses": 5,
    "goals_for": 16,
    "goals_against": 19,
    "goal_difference": -3,
    "points": 14,
    "recent_results": ["V", "D", "E", "V", "D"],
},
{
    "position": 14,
    "team": "Nacional",
    "games_played": 13,
    "wins": 3,
    "draws": 3,
    "losses": 7,
    "goals_for": 12,
    "goals_against": 18,
    "goal_difference": -6,
    "points": 12,
    "recent_results": ["D", "D", "E", "D", "E"],
},
{
    "position": 15,
    "team": "Casa Pia AC",
    "games_played": 13,
    "wins": 2,
    "draws": 3,
    "losses": 8,
    "goals_for": 13,
    "goals_against": 27,
    "goal_difference": -14,
    "points": 9,
    "recent_results": ["D", "D", "E", "D", "D"],
},
{
    "position": 16,
    "team": "Tondela",
    "games_played": 13,
    "wins": 2,
    "draws": 3,
    "losses": 8,
    "goals_for": 7,
    "goals_against": 22,
    "goal_difference": -15,
    "points": 9,
    "recent_results": ["D", "V", "D", "E", "D"],
},
{
    "position": 17,
    "team": "Arouca",
    "games_played": 13,
    "wins": 2,
    "draws": 3,
    "losses": 8,
    "goals_for": 14,
    "goals_against": 37,
    "goal_difference": -23,
    "points": 9,
    "recent_results": ["D", "D", "D", "D", "D"],
},
{
    "position": 18,
    "team": "AFS",
    "games_played": 13,
    "wins": 0,
    "draws": 3,
    "losses": 10,
    "goals_for": 9,
    "goals_against": 31,
    "goal_difference": -22,
    "points": 3,
    "recent_results": ["D", "D", "E", "E", "D"],
}]


In [ ]:
# ========== 初始化 OpenAI 客户端 ==========

# 创建客户端：默认从环境变量 OPENAI_API_KEY 读取密钥
openai = OpenAI()


In [ ]:
# ========== 第 1 步：创建 system / user 提示词 ==========

# system_prompt：设定「葡萄牙超级联赛体育分析师」角色；英文原文勿改译
system_prompt = """
You are a sport analyst that provides insights on sports events and players, 
summarizing the latest news and statistics in a concise manner for the 
Portuguese championship - Primeira Liga."""


# user_prompt：用户问题（问前三名球队统计）；英文原文保持
user_prompt = """
    can you tell me the the statistics of the top 3 teams in the Primeira Liga?
"""


# ========== 第 2 步：组装 messages 列表（system + user）==========
messages = [
    # system：角色与输出风格
    {"role": "system", "content": system_prompt}, 
    # user：问题文本 + 换行 + 把整个 stats 列表转成字符串塞进上下文
    {"role": "user", "content": user_prompt + "\n" + str(stats)}
    ]

# ========== 第 3 步：调用 OpenAI Chat Completions ==========
# model 用 gpt-5-nano；messages 就是上面组装好的对话
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)

# ========== 第 4 步：取出回复并渲染为 Markdown ==========
# choices[0].message.content：第一条候选的正文；display 负责在笔记本里展示
# 外面再 print(...)：会多打印一个 display 的返回值（通常是 None），逻辑保持原样不改
print(display(Markdown(response.choices[0].message.content)))
